# BlackCode — entraînement sur T4

Ce notebook exécute le pipeline complet :

1. vérification du GPU et installation ;
2. préparation d'un corpus de code tokenisé (BPE byte-level, 32k) ;
3. validation rapide avec le modèle `nano` ;
4. entraînement de `t4_small` (51 M paramètres, ternaire, hybride 3 conv / 1 attention) avec sauvegarde sur Drive et reprise ;
5. export au format `.bcpkg` + budget RAM ;
6. génération et tests.

**Attention** : un T4 gratuit ne permet pas d'entraîner 1 Md de paramètres de zéro. Ce notebook sert à valider l'architecture et à produire un modèle de recherche de 51 M. La distillation du modèle de 1,2 Md est décrite dans `docs/04-PROTOCOLE-T4.md`.


In [ ]:
#@title 1. Vérification du GPU et installation
import os, subprocess, sys

REPO_URL = "https://github.com/<votre-compte>/Llm-search.git"  # <-- à remplacer
REPO = "/content/Llm-search"

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
os.chdir(REPO)
print("dépôt :", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("dépendances installées")


In [ ]:
#@title 2. Monter Google Drive (optionnel, recommandé)
OUT = "/content/runs/t4_small"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/blackcode/runs/t4_small"
except Exception as exc:
    print("Drive non monté, sauvegarde locale :", exc)
os.makedirs(OUT, exist_ok=True)
print("sortie :", OUT)


## 3. Corpus

Le notebook clone quelques dépôts GitHub : c'est rapide et suffisant pour valider le pipeline (quelques millions de tokens).

Pour un vrai entraînement (1 Md de tokens), utiliser `bigcode/the-stack-smol` ou `codeparrot/codeparrot-clean` converti en JSONL — voir `docs/04-PROTOCOLE-T4.md`, section 1.


In [ ]:
#@title 4. Préparer le corpus
os.makedirs("/content/corpus", exist_ok=True)
repos = {
    "algorithms": "https://github.com/TheAlgorithms/Python.git",
    "requests": "https://github.com/psf/requests.git",
    "flask": "https://github.com/pallets/flask.git",
}
for name, url in repos.items():
    dst = f"/content/corpus/{name}"
    if not os.path.isdir(dst):
        subprocess.run(["git", "clone", "--depth", "1", url, dst], check=True)

subprocess.run([
    sys.executable, "-m", "blackcode.corpus", "prepare",
    "--src", "/content/corpus", "--out", "data/code",
    "--vocab", "32000", "--suffixes", ".py,.pyi",
    "--max-bytes", "200000",
], check=True)


In [ ]:
#@title 5. Validation rapide : nano (2,2 M paramètres), ~2 minutes
subprocess.run([
    sys.executable, "-m", "blackcode.train",
    "--config", "configs/nano.json", "--data", "data/code",
    "--out", "runs/nano_colab", "--batch-size", "8", "--seq-len", "128",
    "--max-steps", "300", "--lr", "1.5e-3",
    "--eval-interval", "100", "--save-interval", "100",
], check=True)


## 6. Entraînement `t4_small` (51 M)

`MAX_STEPS = 3000` correspond à ~98 M tokens, environ 30–60 min sur T4.

Pour atteindre 1 Md de tokens : relancer la cellule plusieurs fois (elle reprend grâce à `--resume`), ou passer `MAX_STEPS` à 30000 en acceptant plusieurs sessions. Le checkpoint est écrit toutes les 500 étapes sur Drive.


In [ ]:
#@title 7. Entraîner t4_small (reprise automatique)
MAX_STEPS = 3000
cmd = [
    sys.executable, "-m", "blackcode.train",
    "--config", "configs/t4_small.json", "--data", "data/code", "--out", OUT,
    "--batch-size", "8", "--grad-accum", "4", "--seq-len", "1024",
    "--max-steps", str(MAX_STEPS), "--lr", "6e-4", "--min-lr", "6e-5",
    "--warmup", "200", "--eval-interval", "500", "--save-interval", "500",
    "--grad-checkpoint",
]
if os.path.exists(f"{OUT}/ckpt.pt"):
    cmd.append("--resume")
    print("reprise du checkpoint existant")
subprocess.run(cmd, check=True)


In [ ]:
#@title 8. Export .bcpkg et budget RAM
best = f"{OUT}/best.pt"
if not os.path.exists(best):
    best = f"{OUT}/final.pt"
os.makedirs("artifacts", exist_ok=True)
subprocess.run([
    sys.executable, "-m", "blackcode.export",
    "--ckpt", best, "--out", "artifacts/blackcode_t4.bcpkg", "--context", "4096",
], check=True)

# Budget de la cible 1,19 Md
subprocess.run([
    sys.executable, "-m", "blackcode.planner",
    "--config", "configs/blackcode_1b.json", "--context", "8192",
], check=True)


In [ ]:
#@title 9. Génération
subprocess.run([
    sys.executable, "-m", "blackcode.generate",
    "--ckpt", best, "--tokenizer", "data/code/tokenizer.json",
    "--prompt", "def fibonacci(n):\n", "--max-new-tokens", "120",
    "--temperature", "0.7", "--top-k", "40",
], check=True)


In [ ]:
#@title 10. Tests unitaires
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Et ensuite ?

- **Phase 1** : ablations A1–A6 (ternaire vs fp16, 3:1 vs tout-attention, profondeur, embeddings liés, loop, vocabulaire). Voir `docs/02-ARCHITECTURE.md`.
- **Phase 2** : distillation d'un professeur code (Qwen2.5-Coder-7B en 4 bits) vers un étudiant ternaire de 0,6–1,2 Md. Voir `docs/03-METHODES-NOUVELLES.md` (TAD) et `docs/04-PROTOCOLE-T4.md`.
- **Phase 3** : RLVR sur tests unitaires, best-of-N avec tests, réparateur REN.
- **Phase 4** : moteur C pour tenir sous 1 Go **mesuré** (PyTorch ajoute 300–900 Mo).

**Rappel honnête** : un modèle de 51 M entraîné sur quelques millions de tokens ne code pas. Ce notebook valide la mécanique ; les résultats intéressants commencent avec la distillation et plusieurs milliards de tokens.
